# 05: 量子位相推定 (QPE) — 回路の実装

ノート `05_quantum_phase_estimation.md` の QPE 回路を Qiskit で実装し、シミュレーションで検証する。

**内容:**
1. 2ビット QPE 回路（具体例: $\varphi = 1/4$）
2. 一般的な QPE 回路
3. 位相が $n$ ビットで正確に表せない場合の振る舞い

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Statevector, Operator
from qiskit.circuit.library import QFT
import numpy as np

## 1. 2ビット QPE（$\varphi = 1/4$）

![2ビット QPE 回路](../images/05_qpe_2bit.png)

ノート05の具体例と同じ設定で QPE を実装する。

- **ユニタリ演算子 $U$:** 位相ゲート $R_\varphi = \begin{pmatrix} 1 & 0 \\ 0 & e^{2\pi i \varphi} \end{pmatrix}$
- **固有ベクトル:** $|u\rangle = |1\rangle$（固有値 $e^{2\pi i \varphi}$）
- **位相:** $\varphi = 1/4$
- **計数レジスタ:** $n = 2$ ビット（$2^n = 4$）
- **期待される測定結果:** $s = \varphi \cdot 2^n = 1/4 \cdot 4 = 1$ → 二進数で $01$

In [ ]:
def phase_gate(phi):
    """位相ゲート R_φ = diag(1, e^{2πiφ}) を返す。"""
    qc = QuantumCircuit(1, name=f"R({phi})")
    qc.p(2 * np.pi * phi, 0)
    return qc.to_gate()


def qpe_2bit(phi):
    """2ビット QPE 回路を構築する。
    
    U = R_φ, 固有ベクトル |u⟩ = |1⟩ とする。
    """
    counting = QuantumRegister(2, "counting")
    eigen = QuantumRegister(1, "eigen")
    c = ClassicalRegister(2, "result")
    qc = QuantumCircuit(counting, eigen, c)

    # 固有ベクトル |1⟩ を準備
    qc.x(eigen[0])
    qc.barrier(label="init")

    # ステップ 1: アダマール
    qc.h(counting[0])
    qc.h(counting[1])
    qc.barrier(label="H")

    # ステップ 2: 制御ユニタリ
    # counting_0（最上位ビット）が制御 U^{2^1} = U^2
    c_u2 = phase_gate(phi).power(2).control(1)
    qc.append(c_u2, [counting[0], eigen[0]])

    # counting_1（最下位ビット）が制御 U^{2^0} = U^1
    c_u1 = phase_gate(phi).control(1)
    qc.append(c_u1, [counting[1], eigen[0]])
    qc.barrier(label="CU")

    # ステップ 3: 逆 QFT
    qft_inv = QFT(2, inverse=True).to_gate()
    qft_inv.label = "QFT†"
    qc.append(qft_inv, [counting[0], counting[1]])
    qc.barrier(label="QFT†")

    # ステップ 4: 測定
    qc.measure(counting, c)

    return qc


qc_2bit = qpe_2bit(1/4)
qc_2bit.draw("mpl", fold=-1)

### 状態ベクトルで各ステップを追跡

ノート05のステップ 1〜3 に対応する状態の変化を確認する。

In [ ]:
phi = 1/4

# 測定なしの回路で状態ベクトルを追跡
def qpe_2bit_no_measure(phi):
    """測定なしの 2ビット QPE 回路（状態ベクトル取得用）。"""
    qc = QuantumCircuit(3)  # q0=counting_0, q1=counting_1, q2=eigen

    # 固有ベクトル |1⟩
    qc.x(2)

    # ステップ 1: アダマール
    qc.h(0)
    qc.h(1)

    # ステップ 2: 制御ユニタリ
    # counting_0 が制御 U^2
    qc.cp(2 * np.pi * phi * 2, 0, 2)
    # counting_1 が制御 U^1
    qc.cp(2 * np.pi * phi, 1, 2)

    # ステップ 3: 逆 QFT
    qft_inv = QFT(2, inverse=True).to_gate()
    qc.append(qft_inv, [0, 1])

    return qc


qc_track = qpe_2bit_no_measure(phi)
sv = Statevector.from_instruction(qc_track)

print(f"位相 φ = {phi}")
print(f"期待される測定結果: s = {phi} × 2² = {int(phi * 4)}\n")
print("最終状態の振幅:")
for i, amp in enumerate(sv):
    if abs(amp) > 1e-10:
        # q0=counting_0, q1=counting_1, q2=eigen
        c0 = (i >> 0) & 1  # counting_0
        c1 = (i >> 1) & 1  # counting_1
        e = (i >> 2) & 1   # eigen
        counting_val = c0 * 2 + c1  # counting_0 が最上位
        print(f"  |counting={c0}{c1}⟩|eigen={e}⟩ (s={counting_val}): {amp:.4f}")

print(f"\n→ 計数レジスタの測定結果: s = {int(phi * 4)}, φ̃ = s/2² = {int(phi * 4)}/4 = {phi}")

## 2. 一般的な QPE 回路

![一般的な QPE 回路](../images/05_qpe_general.png)

計数レジスタのビット数 $n$ を変えて QPE を実行する。

In [ ]:
def qpe_general(phi, n_counting):
    """一般的な QPE 回路を構築する（測定なし）。
    
    U = R_φ, 固有ベクトル |u⟩ = |1⟩ とする。
    n_counting: 計数レジスタのビット数
    """
    n_total = n_counting + 1  # +1 for eigen register
    qc = QuantumCircuit(n_total)

    # 固有ベクトル |1⟩
    qc.x(n_counting)

    # ステップ 1: アダマール
    for i in range(n_counting):
        qc.h(i)

    # ステップ 2: 制御ユニタリ
    # qubit k（k=0 が最上位）が制御 U^{2^{n-1-k}}
    for k in range(n_counting):
        power = 2 ** (n_counting - 1 - k)
        qc.cp(2 * np.pi * phi * power, k, n_counting)

    # ステップ 3: 逆 QFT
    qft_inv = QFT(n_counting, inverse=True).to_gate()
    qc.append(qft_inv, range(n_counting))

    return qc


def run_qpe(phi, n_counting):
    """QPE を実行し、測定確率を表示する。"""
    qc = qpe_general(phi, n_counting)
    sv = Statevector.from_instruction(qc)
    probs = {}

    for i, amp in enumerate(sv):
        prob = abs(amp) ** 2
        if prob > 1e-10:
            # 計数レジスタの値を取得
            bits = []
            for k in range(n_counting):
                bits.append((i >> k) & 1)
            # bits[0] が qubit 0（最上位）
            s = sum(bits[k] * 2 ** (n_counting - 1 - k) for k in range(n_counting))
            if s not in probs:
                probs[s] = 0.0
            probs[s] += prob

    N = 2 ** n_counting
    print(f"φ = {phi}, n = {n_counting} ビット (2^n = {N})")
    print(f"φ × 2^n = {phi * N:.4f}")
    print(f"{'s':>5}  {'s/2^n':>10}  {'確率':>10}")
    print("-" * 30)
    for s in sorted(probs.keys()):
        if probs[s] > 1e-10:
            print(f"{s:5d}  {s/N:10.6f}  {probs[s]:10.6f}")
    print()
    return probs

### 2.1 位相が $n$ ビットで正確に表せる場合

$\varphi = 1/4$ は $n = 2$ ビットで正確に表せる（$s = 1$）。確率 1 で正しい結果が得られるはずである。

In [ ]:
# φ = 1/4, n = 2: 正確に表せる
run_qpe(1/4, 2)

# φ = 3/8, n = 3: 正確に表せる (s = 3)
run_qpe(3/8, 3)

### 2.2 位相が $n$ ビットで正確に表せない場合

$\varphi = 1/3$ は二進小数で無限に続くため、有限ビットでは正確に表せない。ノート05で説明した通り、最も近い値が最大確率で得られるが、他の値も確率を持つ。

ビット数 $n$ を増やすと精度が向上する様子を確認する。

In [ ]:
# φ = 1/3 を n = 3, 4, 6 ビットで推定
for n in [3, 4, 6]:
    probs = run_qpe(1/3, n)
    # 最大確率の s を見つける
    best_s = max(probs, key=probs.get)
    N = 2 ** n
    print(f"  → 最良推定: s = {best_s}, φ̃ = {best_s}/{N} = {best_s/N:.6f}")
    print(f"     真の値との誤差: |φ̃ - φ| = {abs(best_s/N - 1/3):.6f}")
    print()

### 2.3 固有ベクトルが未知の場合

ノート05の「固有ベクトルが未知の場合」で説明した通り、固有ベクトルの重ね合わせを入力すると、各固有値に対応する位相がランダムに（確率的に）得られる。

$U = R_\varphi$ の場合、$|0\rangle$ と $|1\rangle$ が固有ベクトルであり：
- $U|0\rangle = |0\rangle$（固有値 1、位相 0）
- $U|1\rangle = e^{2\pi i\varphi}|1\rangle$（固有値 $e^{2\pi i\varphi}$、位相 $\varphi$）

$|+\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}$ を入力すると、位相 0 と $\varphi$ がそれぞれ確率 $1/2$ で得られるはずである。

In [ ]:
def qpe_with_plus_input(phi, n_counting):
    """固有ベクトルの代わりに |+⟩ を入力する QPE。"""
    n_total = n_counting + 1
    qc = QuantumCircuit(n_total)

    # |+⟩ = H|0⟩ を準備（|0⟩ と |1⟩ の重ね合わせ）
    qc.h(n_counting)

    # ステップ 1: アダマール
    for i in range(n_counting):
        qc.h(i)

    # ステップ 2: 制御ユニタリ
    for k in range(n_counting):
        power = 2 ** (n_counting - 1 - k)
        qc.cp(2 * np.pi * phi * power, k, n_counting)

    # ステップ 3: 逆 QFT
    qft_inv = QFT(n_counting, inverse=True).to_gate()
    qc.append(qft_inv, range(n_counting))

    return qc


phi = 1/4
n = 2
qc = qpe_with_plus_input(phi, n)
sv = Statevector.from_instruction(qc)

print(f"固有ベクトル |+⟩ を入力した QPE (φ = {phi}, n = {n})")
print()
probs = {}
for i, amp in enumerate(sv):
    prob = abs(amp) ** 2
    if prob > 1e-10:
        bits = [(i >> k) & 1 for k in range(n)]
        s = sum(bits[k] * 2 ** (n - 1 - k) for k in range(n))
        e = (i >> n) & 1
        key = (s, e)
        if key not in probs:
            probs[key] = 0.0
        probs[key] += prob

N = 2 ** n
print(f"{'s':>5}  {'eigen':>5}  {'s/2^n':>10}  {'確率':>10}")
print("-" * 35)
for (s, e) in sorted(probs.keys()):
    print(f"{s:5d}  {e:5d}  {s/N:10.6f}  {probs[(s,e)]:10.6f}")

print(f"\n→ s=0 (位相 0, 固有値 1) と s=1 (位相 1/4) がそれぞれ確率 1/2 で出現")